# DESC tomographic sigma8-bias metric versus the E(B-V) footprint threshold

- Author: Sylvie Dagoret-Campagne
- Creation date: 2026-09-28
- Kernel: conda_py313_opsim53
- Context: SCOC footprint-shrinking study + DESC metrics. This notebook extends the `10_DESCMAFDEPTHANDGALCOUNT` series (E(B-V) footprint-threshold comparison of `ExgalM5WithCuts`, galaxy counts, weak-lensing proxies, 3x2pt FoM, SN counts, kilonovae) to the **tomographic sigma8-bias metric** of the DESC clustering (large-scale structure) analysis.
- This notebook: **10** of the series - `NestedLinearMultibandModelMetric` (parent, per pixel) + `TomographicClusteringSigma8biasMetric` (summary), recomputed **for every year (1-10) and on each of the 7 dust-threshold footprint variants** of the v5.3.6 simulations.
- Reference notebook (baseline only, year by year): `../02_MAF/science/DESC/01_sigma8tomography_demo.ipynb` (Demo 1 only: the AreaAtRisk and mean-z demos of that notebook are **not** treated here).
- Companion notebooks of this series: `01_compareExgalM5withCuts.ipynb`, `06_FOM3x2pts_DESC_TaskForce_demo.ipynb` (same year-by-year x 7-runs layout, whose structure is reused here).
- OpSim simulations analyzed (`/Users/dagoret/DATA/OpSim/`):
  - `shrink_fp_dust_0.050_v5.3.6_10yrs.db`
  - `shrink_fp_dust_0.080_v5.3.6_10yrs.db`
  - `shrink_fp_dust_0.120_v5.3.6_10yrs.db`
  - `shrink_fp_dust_0.150_v5.3.6_10yrs.db`
  - `shrink_fp_dust_0.199_v5.3.6_10yrs.db`
  - `baseline_v5.3.6_10yrs.db` (E(B-V) threshold 0.200)
  - `shrink_fp_dust_0.250_v5.3.6_10yrs.db`


## Notebook overview

**What the metric does.** Non-uniform depth across the sky imprints a spurious galaxy-density fluctuation, which adds fake large-scale power to the galaxy clustering measured by DESC and therefore biases the inferred amplitude of matter clustering sigma8.
1. The per-pixel (Healpix) *parent* metric `NestedLinearMultibandModelMetric` computes the per-band coadded depth of the pixel (with `ExgalM5WithCuts`-like cuts: `extinction_cut`, `n_filters`) and converts it, with the linear dlogN/dm5 model of the year, into a predicted density fluctuation for each of the 5 DESC tomographic bins.
2. The *summary* metric `TomographicClusteringSigma8biasMetric` computes the angular power spectra of these 5 maps up to a bin-dependent lmax, compares the spurious power with the fiducial DESC theory (`DENSITY_TOMOGRAPHY_MODEL`), and returns a single number: the bias on sigma8 (in units of sigma), assuming that a fraction `PERCENTAGE_UNCORRECTED` of the spurious power is not removed by calibration/masking.
3. This is repeated for each survey year (`night <= year*365.25`, non-DD), because the model coefficients depend on the year.

**What is new with respect to the reference notebook.** The same chain is run on the 7 runs of the series, with two E(B-V)-dependent ingredients that follow the footprint threshold of each run (both editable in Section 3):
- `EBV_CUT_MODE = 'run'` (default): the metric's own dust cut `extinction_cut` is set to the E(B-V) threshold that defines the WFD footprint of the run (0.050, 0.080, 0.120, 0.150, 0.199, 0.200 for the baseline, 0.250). `'fixed'` uses `FIXED_LIM_EBV = 0.2` for every run (the configuration of the reference notebook).
- `FOOTPRINT_MODE = 'run'` (default): the Healpix subset on which the metric is evaluated. The reference notebook uses the `lowdust` region of `SkyAreaGenerator` with its default dust limit, i.e. a **fixed** footprint; here the `lowdust` region is regenerated with `dust_limit` set to the threshold of each run (if the installed `SkyAreaGenerator` has that argument, checked in Section 4), so that the evaluated footprint follows the run. `'default'` reproduces the fixed footprint of the reference notebook.

**How the figures are built** (same presentation as notebook 06: the metric is a scalar per run and per year, not a map):
1. Results table (run x year): sigma8 bias, footprint area, valid area.
2. Bias versus year, one curve per run (color = E(B-V) threshold), raw and normalized to the baseline of the same year.
3. Bias versus E(B-V) threshold, one curve per year, raw and normalized.
4. Heatmap over the full (E(B-V) threshold, year) grid, absolute and ratio to the baseline.
5. Footprint/valid area versus threshold, and bias versus valid area.
6. A consistency check against the reference notebook.

**Caching.** MAF is run once per (run, configuration); the 10 yearly results of a run are cached together as one `.csv` in `data_10_SIGMA8TOMO/` (set `FORCE_RECOMPUTE = True` to redo). Within one run, the 10 yearly bundles are grouped in a single `MetricBundleGroup`, so each OpSim database is read once (7 reads, not 70).

## Practical note before running

7 runs x 10 years = 70 evaluations of the parent metric and of the angular-power-spectrum summary metric at `nside = 64`. The first execution of Section 6 can take a while (the summary metric computes 5 power spectra per year); the cached `.csv` files make every later execution instant. The output directories are created next to this notebook.

## 1. Imports

In [ ]:
import os
import time
import inspect
from os.path import join, isfile

import numpy as np
import pandas as pd
import healpy as hp
import matplotlib.pyplot as plt

import rubin_sim
import rubin_sim.maf as maf
import rubin_sim.maf.metric_bundles as mb
from rubin_sim.maf.metrics.uniformity_metrics import NestedLinearMultibandModelMetric
from rubin_sim.maf.metrics.cosmology_summary_metrics import TomographicClusteringSigma8biasMetric
from rubin_sim.maf.metrics.tomography_models import DENSITY_TOMOGRAPHY_MODEL
from rubin_scheduler.scheduler.utils import SkyAreaGenerator

print("rubin_sim version:", rubin_sim.__version__)

## 2. Output directories

In [ ]:
NB_TAG = "SIGMA8TOMO"
data_dir = f"data_10_{NB_TAG}"
figs_dir = f"figs_10_{NB_TAG}"
os.makedirs(data_dir, exist_ok=True)
os.makedirs(figs_dir, exist_ok=True)
print("MAF output (data) directory :", os.path.abspath(data_dir))
print("Figures output directory    :", os.path.abspath(figs_dir))

resultsDb = maf.db.ResultsDb(out_dir=data_dir)

## 3. Configuration

In [ ]:
# Directory holding the OpSim databases
OPSIM_DIR = "/Users/dagoret/DATA/OpSim"

# (run_name, E(B-V) threshold that defines the footprint of the run), sorted by increasing threshold
# - identical list to the other notebooks of this series
RUNS_INFO = [
    ("shrink_fp_dust_0.050_v5.3.6_10yrs", 0.050),
    ("shrink_fp_dust_0.080_v5.3.6_10yrs", 0.080),
    ("shrink_fp_dust_0.120_v5.3.6_10yrs", 0.120),
    ("shrink_fp_dust_0.150_v5.3.6_10yrs", 0.150),
    ("shrink_fp_dust_0.199_v5.3.6_10yrs", 0.199),
    ("baseline_v5.3.6_10yrs", 0.200),
    ("shrink_fp_dust_0.250_v5.3.6_10yrs", 0.250),
]
RUN_NAMES = [r for r, _ in RUNS_INFO]
DUST_THRESH = dict(RUNS_INFO)
REF_RUN = "baseline_v5.3.6_10yrs"


def get_db_path(run_name):
    # Look in OPSIM_DIR, then in OPSIM_DIR/sim_baseline (where the baseline may be stored)
    fname = run_name + ".db"
    candidates = [join(OPSIM_DIR, fname), join(OPSIM_DIR, "sim_baseline", fname)]
    for c in candidates:
        if isfile(c):
            return c
    raise FileNotFoundError(f"OpSim db not found for run {run_name}. Tried: {candidates}")


for run_name, dust in RUNS_INFO:
    try:
        path = get_db_path(run_name)
    except FileNotFoundError:
        path = "NOT FOUND (only a problem if there is no cached result, see Section 6)"
    print(f"{run_name:40s} E(B-V) < {dust:.3f}  ->  {path}")

In [ ]:
# Parameters copied from the Demo 1 of 01_sigma8tomography_demo.ipynb
NSIDE = 64
N_FILTERS = 6  # minimum number of filters per pixel (cut going into the ExgalM5WithCuts-like depth)
LMIN = 10  # smallest multipole included in the Cell sum
PERCENTAGE_UNCORRECTED = 0.1  # fraction of the spurious depth-driven power assumed NOT removed
CONVERT_TO_SIGMA8 = True  # bias on sigma8 (True) or sigma8^2 (False)
YEARS = list(range(1, 11))
YEARS_SHOWN = [1, 3, 6, 10]  # subset of years drawn in Section 8

# E(B-V) cut applied by the METRIC (extinction_cut of NestedLinearMultibandModelMetric):
#   'run'   -> E(B-V) threshold that defines the WFD footprint of each run (DUST_THRESH)
#   'fixed' -> the same value FIXED_LIM_EBV for all runs (configuration of the reference notebook)
EBV_CUT_MODE = "fixed"
FIXED_LIM_EBV = 0.2

# Healpix footprint on which the metric is evaluated:
#   'run'     -> SkyAreaGenerator 'lowdust' region regenerated with dust_limit = threshold of the run
#                (needs a SkyAreaGenerator with a dust_limit argument, checked in Section 4)
#   'default' -> the fixed default 'lowdust' region, as in the reference notebook
FOOTPRINT_MODE = "run"

FORCE_RECOMPUTE = False  # True -> ignore the cached .csv results and rerun MAF

pix_area = hp.nside2pixarea(NSIDE, degrees=True)
print(f"nside={NSIDE} -> npix={hp.nside2npix(NSIDE)}, pixel area = {pix_area:.4f} deg^2")
print(f"Years evaluated: {YEARS}")


def metric_ebv_cut(run_name):
    # E(B-V) cut given to the metric for this run
    if EBV_CUT_MODE == "run":
        return DUST_THRESH[run_name]
    if EBV_CUT_MODE == "fixed":
        return FIXED_LIM_EBV
    raise ValueError(f"EBV_CUT_MODE must be run or fixed, got {EBV_CUT_MODE!r}")

## 4. Footprint of each run

The reference notebook evaluates the metric on `np.where(map_labels == 'lowdust')` of `SkyAreaGenerator(nside)`, whose dust limit is the scheduler default and does **not** depend on the run. For a threshold study this would leave the footprint unchanged, and only the `extinction_cut` of the parent metric would vary (it can only *shrink* the area, so thresholds above the default would have no effect). With `FOOTPRINT_MODE = 'run'` the footprint is regenerated for each threshold. The cell below checks that the installed `SkyAreaGenerator` accepts `dust_limit`, and the table that follows checks that the footprint really changes with it (**verify this table before trusting the results**).

In [ ]:
HAS_DUST_LIMIT = "dust_limit" in inspect.signature(SkyAreaGenerator.__init__).parameters
print("SkyAreaGenerator accepts dust_limit:", HAS_DUST_LIMIT)
if FOOTPRINT_MODE == "run" and not HAS_DUST_LIMIT:
    print(
        "WARNING: FOOTPRINT_MODE is run but dust_limit is not available -> the default footprint is used for all runs."
    )

_fp_cache = {}


def footprint_dust_limit(run_name):
    # dust_limit given to SkyAreaGenerator for this run (None = scheduler default)
    if FOOTPRINT_MODE == "default":
        return None
    if FOOTPRINT_MODE == "run":
        return DUST_THRESH[run_name] if HAS_DUST_LIMIT else None
    raise ValueError(f"FOOTPRINT_MODE must be run or default, got {FOOTPRINT_MODE!r}")


def footprint_hpid(dust_limit):
    # Healpix ids of the lowdust region (nside = NSIDE), cached per dust_limit
    key = None if dust_limit is None else round(float(dust_limit), 4)
    if key not in _fp_cache:
        if key is None:
            gen = SkyAreaGenerator(nside=NSIDE)
        else:
            gen = SkyAreaGenerator(nside=NSIDE, dust_limit=key)
        footprint_maps, footprint_labels = gen.return_maps()
        _fp_cache[key] = np.where(footprint_labels == "lowdust")[0]
    return _fp_cache[key]

In [ ]:
rows = [{"footprint": "default SkyAreaGenerator", "dust_limit": np.nan, "n_pix": len(footprint_hpid(None))}]
if HAS_DUST_LIMIT:
    for run_name, dust in RUNS_INFO:
        rows.append({"footprint": run_name, "dust_limit": dust, "n_pix": len(footprint_hpid(dust))})
fp_table = pd.DataFrame(rows)
fp_table["area_deg2"] = fp_table["n_pix"] * pix_area
print(fp_table.round(1).to_string(index=False))
if HAS_DUST_LIMIT and fp_table["n_pix"].iloc[1:].nunique() == 1:
    print("WARNING: the footprint does not change with dust_limit; check the SkyAreaGenerator version.")

## 5. Helper functions

`build_bundle` is the chain of `extract_sigma8_tomography_metric` of the reference notebook for one (run, year). Note that the reference function labels its bundles with a global `run_name` variable; here the run name is passed explicitly.

In [ ]:
def run_label(run_name):
    label = f"E(B-V) < {DUST_THRESH[run_name]:.3f}"
    return label + " (baseline)" if run_name.startswith("baseline") else label


def save_fig(fig, name):
    base = join(figs_dir, name)
    fig.savefig(base + ".png", dpi=150, bbox_inches="tight")
    fig.savefig(base + ".pdf", bbox_inches="tight")
    print("Saved:", base + ".png/.pdf")


def config_tag(run_name):
    # Cache tag: it contains every setting that changes the result
    dl = footprint_dust_limit(run_name)
    fp = "default" if dl is None else f"{dl:.3f}"
    return (
        f"nf{N_FILTERS}_ebv{metric_ebv_cut(run_name):.3f}_fp{fp}"
        f"_nside{NSIDE}_lmin{LMIN}_unc{PERCENTAGE_UNCORRECTED}"
    )


def cache_path(run_name):
    tag = run_name.replace(".", "_")
    return join(data_dir, f"sigma8bias_{tag}_{config_tag(run_name)}.csv")


def build_bundle(run_name, year, ebv_cut, hpid):
    # Parent metric (per pixel): depth -> density fluctuation in the 5 tomographic bins.
    # Summary metric (map -> one number): bias on sigma8 from the spurious power spectra.
    model = DENSITY_TOMOGRAPHY_MODEL["year" + str(year)]
    days = int(year * 365.25)
    constraint = (
        f'scheduler_note not like "DD%" and night <= {days} and scheduler_note not like "twilight_near_sun"'
    )
    parent_metric = NestedLinearMultibandModelMetric(
        model["dlogN_dm5"],
        extinction_cut=ebv_cut,
        n_filters=N_FILTERS,
        mean_depth={},
        min_depth_cut={},
        max_depth_cut={},
    )
    summary_metrics = [
        TomographicClusteringSigma8biasMetric(
            model,
            convert_to_sigma8=CONVERT_TO_SIGMA8,
            power_multiplier=PERCENTAGE_UNCORRECTED,
            lmin=LMIN,
        )
    ]
    slicer = maf.HealpixSubsetSlicer(nside=NSIDE, hpid=hpid, use_cache=False)
    return mb.MetricBundle(
        metric=parent_metric,
        slicer=slicer,
        constraint=constraint,
        run_name=run_name,
        info_label=f"year {year} ebv{ebv_cut:.3f}",
        summary_metrics=summary_metrics,
    )


def compute_run(run_name):
    # All years of one run are grouped in a single MetricBundleGroup: the OpSim db is opened once.
    ebv_cut = metric_ebv_cut(run_name)
    hpid = footprint_hpid(footprint_dust_limit(run_name))
    dbpath = get_db_path(run_name)
    print(f"[{run_name}] extinction_cut = {ebv_cut:.3f}, footprint = {len(hpid)} pixels, db = {dbpath}")
    bundles = {year: build_bundle(run_name, year, ebv_cut, hpid) for year in YEARS}
    group = mb.MetricBundleGroup(
        mb.make_bundles_dict_from_list(list(bundles.values())), dbpath, out_dir=data_dir, results_db=resultsDb
    )
    group.run_all()

    rows = []
    for year in YEARS:
        b = bundles[year]
        bias = b.summary_values.get("TomographicClusteringSigma8bias", np.nan)
        try:
            bias = float(bias)
        except (TypeError, ValueError):
            bias = np.nan
        mask = np.ma.getmaskarray(b.metric_values)
        if mask.ndim > 1:
            mask = mask.all(axis=tuple(range(1, mask.ndim)))
        n_valid = int((~mask).sum())
        rows.append(
            {
                "year": year,
                "ebv_cut_metric": ebv_cut,
                "sigma8_bias": bias,
                "n_footprint_pix": len(hpid),
                "n_valid_pix": n_valid,
                "footprint_area_deg2": len(hpid) * pix_area,
                "valid_area_deg2": n_valid * pix_area,
            }
        )
    return pd.DataFrame(rows).set_index("year")


def load_or_compute(run_name):
    path = cache_path(run_name)
    if isfile(path) and not FORCE_RECOMPUTE:
        print(f"[{run_name}] loaded cached results: {path}")
        return pd.read_csv(path, index_col="year")
    df = compute_run(run_name)
    df.to_csv(path)
    print(f"[{run_name}] saved: {path}")
    return df

## 6. Compute (or reload) the sigma8-bias grid: 7 runs x 10 years

In [ ]:
t0 = time.time()
results_by_run = {run_name: load_or_compute(run_name) for run_name, _ in RUNS_INFO}
print(f"Done in {time.time() - t0:.1f} s")

In [ ]:
combined = pd.concat(results_by_run, names=["run", "year"])
combined["ebv_threshold"] = combined.index.get_level_values("run").map(DUST_THRESH)
combined_csv = join(data_dir, "sigma8bias_all_runs_all_years.csv")
combined.to_csv(combined_csv)
print("Saved:", combined_csv)
combined.round(4)

## 7. sigma8 bias versus survey year, one curve per run

Color = E(B-V) threshold of the run (`viridis`, as in the overlay plots of notebooks 01-03 and 06).

In [ ]:
colors = plt.cm.viridis(np.linspace(0.0, 0.95, len(RUNS_INFO)))

fig, ax = plt.subplots(figsize=(8, 6))
for (run_name, dust), c in zip(RUNS_INFO, colors):
    df = results_by_run[run_name]
    ax.plot(df.index, df["sigma8_bias"], marker="o", color=c, label=run_label(run_name))
ax.set_xlabel("Survey year")
ax.set_ylabel("Tomographic sigma8 bias (units of sigma)")
ax.set_title("TomographicClusteringSigma8biasMetric versus year, 7 E(B-V) footprint variants")
ax.grid(alpha=0.3)
ax.legend(fontsize=8)
fig.tight_layout()
save_fig(fig, "sigma8bias_vs_year_allruns")
plt.show()

### 7.1 Same plot, normalized to the baseline of the same year

In [ ]:
ref = results_by_run[REF_RUN]["sigma8_bias"]

fig, ax = plt.subplots(figsize=(8, 6))
for (run_name, dust), c in zip(RUNS_INFO, colors):
    df = results_by_run[run_name]
    ax.plot(df.index, df["sigma8_bias"] / ref, marker="o", color=c, label=run_label(run_name))
ax.axhline(1.0, color="gray", ls="--", linewidth=0.8)
ax.set_xlabel("Survey year")
ax.set_ylabel("sigma8 bias / baseline sigma8 bias, same year")
ax.set_title("Tomographic sigma8 bias versus year, normalized to the baseline")
ax.grid(alpha=0.3)
ax.legend(fontsize=8)
fig.tight_layout()
save_fig(fig, "sigma8bias_vs_year_allruns_norm")
plt.show()

## 8. sigma8 bias versus the E(B-V) footprint threshold, one curve per year

Cross-sections of the same (run, year) grid with the E(B-V) threshold on the x-axis, the natural way to compare with the 'metric versus E(B-V) threshold' plots of notebooks 01-03 and 06. Edit `YEARS_SHOWN` (Section 3) to change the years drawn.

In [ ]:
x = np.array([DUST_THRESH[r] for r in RUN_NAMES])
i_ref = RUN_NAMES.index(REF_RUN)
year_colors = plt.cm.plasma(np.linspace(0.1, 0.9, len(YEARS_SHOWN)))

fig, ax = plt.subplots(figsize=(7, 5.5))
for year, c in zip(YEARS_SHOWN, year_colors):
    y = [results_by_run[r].loc[year, "sigma8_bias"] for r in RUN_NAMES]
    ax.plot(x, y, marker="o", color=c, label=f"year {year}")
ax.axvline(x[i_ref], color="gray", ls="--", linewidth=0.8, label="baseline threshold")
ax.set_xlabel("E(B-V) threshold that defines the footprint of the run")
ax.set_ylabel("Tomographic sigma8 bias (units of sigma)")
ax.set_title("Tomographic sigma8 bias versus E(B-V) footprint threshold")
ax.grid(alpha=0.3)
ax.legend(fontsize=9)
fig.tight_layout()
save_fig(fig, "sigma8bias_vs_dust_threshold_byyear")
plt.show()

### 8.1 Same plot, normalized to the baseline run (same year)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5.5))
for year, c in zip(YEARS_SHOWN, year_colors):
    y = np.array([results_by_run[r].loc[year, "sigma8_bias"] for r in RUN_NAMES])
    ax.plot(x, y / y[i_ref], marker="o", color=c, label=f"year {year}")
ax.axhline(1.0, color="gray", ls="--", linewidth=0.8)
ax.axvline(x[i_ref], color="gray", ls="--", linewidth=0.8, label="baseline")
ax.set_xlabel("E(B-V) threshold that defines the footprint of the run")
ax.set_ylabel("sigma8 bias / baseline sigma8 bias, same year")
ax.set_title("Tomographic sigma8 bias versus E(B-V) threshold, normalized")
ax.grid(alpha=0.3)
ax.legend(fontsize=9)
fig.tight_layout()
save_fig(fig, "sigma8bias_vs_dust_threshold_byyear_norm")
plt.show()

## 9. Heatmap over the full (E(B-V) threshold, year) grid

Left: absolute bias. Right: ratio to the baseline row (year by year). Rows are sorted by increasing E(B-V) threshold, columns by year.

In [ ]:
grid = np.array([[results_by_run[r].loc[y, "sigma8_bias"] for y in YEARS] for r in RUN_NAMES])
with np.errstate(divide="ignore", invalid="ignore"):
    grid_norm = grid / grid[i_ref : i_ref + 1, :]
dev = np.nanmax(np.abs(grid_norm - 1.0))

fig, axs = plt.subplots(1, 2, figsize=(13, 5))
im0 = axs[0].imshow(grid, aspect="auto", cmap="viridis", origin="lower")
axs[0].set_title("sigma8 bias (units of sigma)")
fig.colorbar(im0, ax=axs[0], label="bias")
im1 = axs[1].imshow(grid_norm, aspect="auto", cmap="RdBu_r", origin="lower", vmin=1 - dev, vmax=1 + dev)
axs[1].set_title("sigma8 bias / baseline (same year)")
fig.colorbar(im1, ax=axs[1], label="ratio")

for ax in axs:
    ax.set_xticks(range(len(YEARS)))
    ax.set_xticklabels(YEARS)
    ax.set_xlabel("Survey year")
    ax.set_yticks(range(len(RUN_NAMES)))
    ax.set_yticklabels([f"{DUST_THRESH[r]:.3f}" for r in RUN_NAMES])
    ax.set_ylabel("E(B-V) threshold")
    ax.axhline(i_ref, color="k", linewidth=0.8, linestyle="--")

fig.suptitle("Tomographic sigma8 bias over the full (E(B-V) threshold, year) grid", y=1.03)
fig.tight_layout()
save_fig(fig, "sigma8bias_heatmap_dustthreshold_vs_year")
plt.show()

## 10. Footprint area and sigma8 bias

Left: area of the Healpix footprint given to the slicer, and area of the pixels where the parent metric is actually valid (year 10; the parent metric masks the pixels failing the `extinction_cut`, `n_filters` or depth conditions). Right: year-10 bias versus valid area, each point labeled with the E(B-V) threshold.

In [ ]:
fp_area10 = np.array([results_by_run[r].loc[10, "footprint_area_deg2"] for r in RUN_NAMES])
valid_area10 = np.array([results_by_run[r].loc[10, "valid_area_deg2"] for r in RUN_NAMES])
bias10 = np.array([results_by_run[r].loc[10, "sigma8_bias"] for r in RUN_NAMES])

fig, axs = plt.subplots(1, 2, figsize=(12, 5))
axs[0].plot(x, fp_area10, marker="s", label="footprint given to the slicer")
axs[0].plot(x, valid_area10, marker="o", label="valid pixels of the parent metric (year 10)")
axs[0].axvline(x[i_ref], color="gray", ls="--", linewidth=0.8)
axs[0].set_xlabel("E(B-V) threshold")
axs[0].set_ylabel("Area (deg^2)")
axs[0].grid(alpha=0.3)
axs[0].legend(fontsize=9)

axs[1].scatter(valid_area10, bias10, color=colors, s=60)
for xi, ai, bi in zip(x, valid_area10, bias10):
    axs[1].annotate(f"{xi:.3f}", (ai, bi), textcoords="offset points", xytext=(5, 5), fontsize=8)
axs[1].set_xlabel("Valid area of the parent metric, year 10 (deg^2)")
axs[1].set_ylabel("Tomographic sigma8 bias, year 10 (units of sigma)")
axs[1].grid(alpha=0.3)

fig.tight_layout()
save_fig(fig, "sigma8bias_area_vs_dust_threshold")
plt.show()

## 11. Consistency check against the reference notebook

The baseline row below can be compared with the curve of `../02_MAF/science/DESC/01_sigma8tomography_demo.ipynb` (Demo 1). An exact match needs the configuration of that notebook: `EBV_CUT_MODE = 'fixed'` (0.2), `FOOTPRINT_MODE = 'default'`, and the same OpSim file. Two differences remain even then: the reference notebook analyzes `baseline_v5.3.6_10yrs` whereas this series uses `baseline_v5.3.6_11yrs`, and the reference notebook labels its bundles with a global run name (cosmetic only).

In [ ]:
print(f"Baseline run: {REF_RUN}")
print(f"EBV_CUT_MODE = {EBV_CUT_MODE}, FOOTPRINT_MODE = {FOOTPRINT_MODE}")
results_by_run[REF_RUN][["sigma8_bias", "footprint_area_deg2", "valid_area_deg2"]].round(4)

## Caveats

- **Footprint definition.** The whole study rests on `SkyAreaGenerator(dust_limit=...)` reproducing the WFD footprint of each `shrink_fp_dust` run. This is an approximation (the scheduler footprint of these simulations is built with additional smoothing and other regions); check the table of Section 4 and, if needed, compare the valid area with the map of notebook 01 or with the visit map of `../07_variateEVmV`. `FOOTPRINT_MODE = 'default'` isolates the effect of the metric's `extinction_cut` alone.
- **Model coefficients are not footprint-dependent.** `DENSITY_TOMOGRAPHY_MODEL` (dlogN/dm5, Cell fits, lmax) was derived for the nominal footprint and its galaxy sample. Changing the footprint changes only the depth maps fed to the model, not the model itself.
- **sky fraction.** The summary metric computes power spectra of a partial-sky map; as noted by its authors in the reference notebook ('should probably also return fsky'), the treatment of the sky fraction is simplified, so part of the change of the bias with the threshold may come from the shrinking area rather than from the depth uniformity. The right panel of Section 10 helps to separate the two.
- **Multiple ingredients change with the threshold**: the footprint, the visit distribution produced by the scheduler and the metric's own dust cut. `0.199` and `baseline (0.200)` have almost the same nominal footprint: the difference between their curves gives the scale of the run-to-run (scheduler) fluctuations at fixed threshold, to compare with the change between the more widely separated thresholds.
- **Only Demo 1 of the reference notebook is reproduced.** The AreaAtRisk (`UniformAreaFoMFractionMetric`) and mean-redshift (`MultibandMeanzBiasMetric`) demos can be added with the same structure.

## References
- Reference notebook: `../02_MAF/science/DESC/01_sigma8tomography_demo.ipynb`, adapted from https://github.com/lsst/rubin_sim_notebooks/blob/main/maf/science/sigma8tomography_demo.ipynb
- Lochner, M. et al. 2018, 'Optimizing LSST Observing Strategy for Dark Energy Science', arXiv:1808.00006
- Awan, H. et al. 2016, ApJ 829, 50, 'Testing LSST Dither Strategies for Survey Uniformity and Large-Scale Structure Systematics'
- Bianco, F. B. et al. 2022, ApJS 258, 1 (SCOC context)
- `rubin_sim` source: https://github.com/lsst/rubin_sim (`rubin_sim/maf/metrics/uniformity_metrics.py`, `cosmology_summary_metrics.py`, `tomography_models.py`)
- `06_FOM3x2pts_DESC_TaskForce_demo.ipynb`, `01_compareExgalM5withCuts.ipynb` - structure and conventions of this series.
- `shrink_fp_dust_*.db` simulations: https://s3df.slac.stanford.edu/data/rubin/sim-data/sims_featureScheduler_runs5.3/shrink_fp/